In [ ]:
from typing import Optional, Tuple, Any
import torch
import torch.nn as nn


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0.0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0.0

    def calculate(self) -> float:
        return self.avg


def print_variance(name: str, data: torch.Tensor):
    # Compute variance across features/neurons and average across the batch
    neuron_variance = torch.mean(torch.var(data.detach().float(), dim=-1))
    print(f"{name}: Variance = {neuron_variance.item():.6f}")


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.scale = n_hidden**-0.5

        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.w_o = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape

        # Shape: (B, T, 3 * num_heads * n_hidden) -> (B, num_heads, T, 3 * n_hidden)
        qkv = (
            self.qkv_projection(x)
            .reshape(B, T, self.num_heads, 3 * self.n_hidden)
            .transpose(1, 2)
        )
        q, k, v = qkv.chunk(3, dim=-1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if attn_mask is not None:
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)
            scores = scores.masked_fill(attn_mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        output = self.w_o(context)
        return output, attn_weights


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)

        # LayerNorm applied inside the FFN sequence only
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Attention block with residual connection (no norm)
        attn_out, alphas = self.attn(x, attn_mask=attn_mask)
        x = x + attn_out

        # FFN block with residual connection (norm is first layer inside self.ffn)
        x = x + self.ffn(x)
        return x, alphas


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        collected_attns = []

        for layer in self.layers:
            x, alphas = layer(x, attn_mask=attn_mask)
            if return_attn:
                collected_attns.append(alphas)

        if return_attn:
            return x, torch.stack(collected_attns, dim=1)
        return x, None


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert (
            img_size % patch_size == 0
        ), "Image dimensions must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Output: (B, num_patches, nout) in float
        return self.proj(x).flatten(2).transpose(1, 2)


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, dim))

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification Head with its own LayerNorm
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # Prepend [CLS] token: (B, num_patches + 1, dim)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, embs), dim=1)

        # Add position embeddings
        x = x + self.pos_embed

        # Pass through Transformer encoder layers
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # Classify using the [CLS] token at index 0
        out = self.head(x[:, 0])

        return out, alphas


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        num_global_tokens: int = 1,  # Number of global/CLS tokens
    ):
        super().__init__()
        self.num_global_tokens = num_global_tokens

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)
        self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))

        # Position embeddings covering both global tokens and image patches
        total_seq_len = num_global_tokens + num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))

        nn.init.trunc_normal_(self.cls_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification / Projection Head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # 1. Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
        cls_tokens = self.cls_tokens.expand(B, -1, -1)

        # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
        x = torch.cat((cls_tokens, embs), dim=1)

        # 4. Add position embeddings
        x = x + self.pos_embed

        # 5. Transformer forward pass
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
        global_repr = x[:, : self.num_global_tokens]

        # Aggregate global tokens:
        # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
        # out = self.head(global_repr[:, 0])
        #
        # - Option B: Mean-pool across all global tokens
        out = self.head(global_repr.mean(dim=1))  # (B, nout)

        return out, alphas